# Week 4: Repair a Messy Prediction Workflow

[Open in Colab](https://colab.research.google.com/github/astarryknight/ai-ml/blob/main/notebooks/week-04-data-pipeline.ipynb)

**Estimated time:** 75–90 minutes  
**Finish line:** turn a messy customer table into a repeatable prediction pipeline, then use it for three new customers.

Machine-learning libraries handle the calculations. Our job is to make the data clear, keep the test set separate, and check whether the result is useful.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Meet the messy customer table

Each row describes a fictional subscription customer. Some values are missing, `plan` contains words, and the numeric columns use different ranges.

In [ ]:
rng = np.random.default_rng(42)
n = 420

customers = pd.DataFrame({
    "customer_id": [f"C{i:04d}" for i in range(n)],
    "sessions_per_week": rng.poisson(4.5, n).astype(float),
    "support_tickets": rng.poisson(1.3, n).astype(float),
    "monthly_spend": rng.normal(42, 12, n).round(2),
    "plan": rng.choice(["basic", "plus", "pro"], n, p=[0.5, 0.35, 0.15]),
})

renewal_signal = (
    0.55 * customers["sessions_per_week"]
    - 0.75 * customers["support_tickets"]
    + 0.03 * customers["monthly_spend"]
    + customers["plan"].map({"basic": -0.4, "plus": 0.2, "pro": 0.7})
    + rng.normal(0, 1.2, n)
)
customers["renewed"] = (renewal_signal > 2.6).astype(int)

for column in ["sessions_per_week", "monthly_spend", "plan"]:
    missing_rows = rng.choice(customers.index, size=18, replace=False)
    customers.loc[missing_rows, column] = np.nan

customers.sample(8, random_state=7)

## 3. Data detective challenge

Before writing model code, identify what the computer cannot use directly. Look for missing values, text categories, identifiers, and columns with different scales.

In [ ]:
audit = pd.DataFrame({
    "type": customers.dtypes.astype(str),
    "missing": customers.isna().sum(),
    "unique": customers.nunique(dropna=True),
})
audit

## 4. Create one useful feature

A customer with six sessions and no support tickets behaves differently from a customer with six sessions and five tickets. This derived feature expresses that relationship. It uses values from one row only, so it does not learn anything from the test set.

In [ ]:
def add_customer_features(frame):
    result = frame.copy()
    result["sessions_per_ticket"] = (
        result["sessions_per_week"] / (1 + result["support_tickets"])
    )
    return result

model_data = add_customer_features(customers)
model_data[["sessions_per_week", "support_tickets", "sessions_per_ticket"]].head()

## 5. Split before learning cleanup values

The test set represents future customers. Medians, scales, and category mappings must come from the training set.

In [ ]:
X = model_data.drop(columns=["customer_id", "renewed"])
y = model_data["renewed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 6. Establish a simple baseline

The baseline always predicts the most common class. A useful model should improve on this reference point.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_f1 = f1_score(y_test, baseline_predictions, zero_division=0)

print(f"Baseline accuracy: {baseline_accuracy:.3f}")
print(f"Baseline F1:       {baseline_f1:.3f}")

## 7. Put every repair inside one pipeline

Numeric columns receive a median and scaling. The plan column receives a fill value and one-hot encoding. The pipeline remembers the same steps for future customers.

In [ ]:
numeric_features = [
    "sessions_per_week", "support_tickets",
    "monthly_spend", "sessions_per_ticket",
]
categorical_features = ["plan"]

numeric_pipeline = Pipeline([
    ("fill_missing", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("fill_missing", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numbers", numeric_pipeline, numeric_features),
    ("categories", categorical_pipeline, categorical_features),
])
model = Pipeline([
    ("prepare", preprocessor),
    ("predict", LogisticRegression(max_iter=1000, random_state=42)),
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)

## 8. Compare the repaired workflow with the baseline

In [ ]:
model_accuracy = accuracy_score(y_test, predictions)
model_f1 = f1_score(y_test, predictions)

comparison = pd.DataFrame({
    "model": ["Most-common baseline", "Repaired pipeline"],
    "accuracy": [baseline_accuracy, model_accuracy],
    "f1": [baseline_f1, model_f1],
})
display(comparison)

comparison.set_index("model").plot(kind="bar", figsize=(8, 4), color=["#8A9AA6", "#147D88"])
plt.ylim(0, 1)
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.title("Did the repaired workflow beat the baseline?")
plt.show()

ConfusionMatrixDisplay.from_predictions(y_test, predictions, display_labels=["Did not renew", "Renewed"], cmap="Blues")
plt.title("Where the pipeline makes mistakes")
plt.show()

## 9. Predict three new customers

The same pipeline accepts raw values. It fills Sam's missing spending value automatically.

In [ ]:
new_customers = pd.DataFrame({
    "name": ["Maya", "Jordan", "Sam"],
    "sessions_per_week": [7, 2, 5],
    "support_tickets": [0, 4, 1],
    "monthly_spend": [58, 24, np.nan],
    "plan": ["pro", "basic", "plus"],
})

new_customer_features = add_customer_features(new_customers)
probabilities = model.predict_proba(new_customer_features.drop(columns="name"))[:, 1]

results = new_customers[["name", "plan"]].copy()
results["renewal_probability"] = probabilities.round(3)
results

## 10. Your experiment

Change one decision and rerun the notebook. Try mean imputation, remove `sessions_per_ticket`, or replace logistic regression with a decision tree. Record the new score and explain whether the change helped.

### Quick check
1. Why does the split happen before preprocessing?
2. Which raw column did we exclude, and why?
3. What information does `sessions_per_ticket` add?

## 11. Validation

In [ ]:
transformed = model.named_steps["prepare"].transform(X_test)
transformed_array = transformed.toarray() if hasattr(transformed, "toarray") else transformed

assert transformed_array.shape[0] == X_test.shape[0]
assert np.isfinite(transformed_array).all()
assert len(results) == 3
assert model_f1 > baseline_f1
print("Validation passed: the workflow handles raw data and predicts new customers.")

## Finish line

- A data-quality audit
- Baseline and pipeline metrics
- Predictions for Maya, Jordan, and Sam
- One changed preprocessing or modeling decision with a short conclusion